[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagedSaeed/instructions-tuning/blob/main/Notebooks/Experiments/Baselines/emotion_detection_tuning.ipynb)

# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [6]:
import sys
sys.path.append('/raid_storage/SLURM/home/slurm_majedalshaibani/Projects/instructions-tuning/jrcai_corekit/llms_corekit')

check everything is working

In [7]:
from llm.text_generator import TextGenerator

/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/deepspeed.py:24: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


In [8]:
TAWJEEH_DATASET_NAME = 'ArEntail'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/ArEntail_experimental'
MODEL_PATH = "/hdd/shared_models/AceGPT-7B"
TASK_NAME='NLI'

In [9]:
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [10]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14892,
  'tags': [],
  'name': 'mais-prompt1',
  'task': {'name': 'NLI'},
  'status': 'SUBMITTED',
  'template': 'Premise: {{ Premise }}\r\nHypothesis: {{ Hypothesis }}\r\n\r\nDoes the hypothesis about the premise entails? (Yes or No)\r\nAnswer:\r\n|||\r\n{{ answer_choices[label] }}',
  'created_by': 'mais',
  'dataset_name': 'arbml/ArabicTE',
  'dataset_subset': 'default',
  'answer_choices': ['No', 'Yes'],
  'text_direction': 'ltr'},
 {'id': 14891,
  'tags': [],
  'name': 'expert Arabic summarizer',
  'task': {'name': 'summarization'},
  'status': 'APPROVED',
  'template': 'You are an expert Arabic text summarizer. The following article:\r\n{{article}}\xa0\r\ncan be summarized as:\r\n|||\r\n{{summary}}',
  'created_by': 'majed.alshaibani',
  'dataset_name': 'arbml/AraSum',
  'dataset_subset': 'default',
  'answer_choices': [],
  'text_direction': 'ltr'},
 {'id': 14890,
  'tags': [],
  'name': 'Translation as completion',
  'task': {'name': 'machine translation'},
  'status': 

In [11]:
len(prompts)

358

In [12]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

235

## Finetuning

### Get the dataset prompts

In [13]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

8

In [14]:
SELECTED_PROMPTS_IDS = [
    14581,   
    14816,
    14818,
    14819,
    14820,
]

In [15]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the experimental dataset from HF

In [16]:
import datasets

In [17]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['premise', 'hypothesis', 'label'],
        num_rows: 1000
    })
})

### Merge the prompts

In [18]:
from jinja2 import Environment, StrictUndefined

In [19]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}\n{suffix.strip()}' # output is always the last line!

In [20]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        sample['answer_choices'] = prompt_template['answer_choices']
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e
        

see how the template is applied on different examples

### Perform prompt-merge on one example prompt, for experimentation

In [21]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][5]))

Welcome to the "Logic Match Game!"

Task: In this game, your challenge is to decide if the statement, "فيروس كورونا: فرنسا تعيد فرض الحجر الشامل لمدة شهر على 16 مقاطعة بينها باريس وضواحيها," logically follows from the premise, "كورونا.. صدامات بين الشرطة ومحتجين بمدن أوروبية وإغلاق شامل في 16 مقاطعة فرنسية." Think carefully: does the premise support the hypothesis, or not?

Rules: 
- If the hypothesis follows logically, answer with "entails".
- If it does not follow, answer with "not entail".

Enter your answer (entails or not entail) to complete the challenge!
entails


In [22]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

1000.0

In [23]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/5000 [00:00<?, ?it/s]

rending Welcome to the "Logic Match Game!"

Task: In this game, your challenge is to decide if the statement, "{{ hypothesis }}," logically follows from the premise, "{{ premise }}." Think carefully: does the premise support the hypothesis, or not?

Rules: 
- If the hypothesis follows logically, answer with "entails".
- If it does not follow, answer with "not entail".

Enter your answer (entails or not entail) to complete the challenge!
|||
{{ answer_choices[label] }} sample index: 0
rending Task: Determine if the statement, "{{ hypothesis }}," logically follows from the premise, "{{ premise }}." Carefully assess whether the premise provides enough support for the hypothesis. 

Answer with "entails" if the hypothesis logically follows, or "not entail" if it does not. Provide only your answer.
|||
{{ answer_choices[label] }} sample index: 1000
rending Task: Determine if the hypothesis follows logically from the premise. Follow these steps to reach a conclusion.

Steps:
1. Understand the

5000

## Finetune the LLM

In [24]:
GLOBAL_SEED = 42

In [25]:
import random
random.seed(GLOBAL_SEED)

In [26]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Llama3Initializer,LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [27]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Llama3Initializer(),
)
llm_loader

In [28]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /hdd/shared_models/AceGPT-7B/config.json
Model config LlamaConfig {
  "_name_or_path": "/hdd/shared_models/AceGPT-7B",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_length": 4096,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 32000
}

loading weights file /hdd/shared_models/AceGPT-7B/pytorch_model.bin
Instantiating LlamaForCausalLM model under default dtype torch.bfl

In [28]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    sample_lines = sample.splitlines()
    # each sample should be two lines only, the first line is for the inputs and the last is the output
    # we did the join for generalization
    prefix = '\n'.join(sample_lines[:-1])
    prefix = prefix.strip()
    prefix += '\nThe answer is:'
    # prefix = re.sub(r'\s+', ' ', prefix).strip()
    # outputs are always the last line
    suffix = sample_lines[-1].strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(4500,
 500,
 [('Based on the premise: منحه فرصة للترشح للرئاسة.. قاض يلغي إجراءات ملاحقة دا سيلفا في قضيتين, does the following hypothesis logically follow?\n\nHypothesis: ألغى قاض في المحكمة العليا البرازيلية إجراءات  ضد الرئيس السابق \n\nAnswer with only "entails" or "not entails". No need for extra explanation\nThe answer is:',
   ' entails'),
  ('Based on the premise: زحف نحو الحدود ومظاهرات لا تتوقف.. الأردن يغضب للقدس وفلسطين ودعوات لقطع العلاقات مع إسرائيل, does the following hypothesis logically follow?\n\nHypothesis: احتجاجات لبنانية قرب الحدود تضامنا مع فلسطين\n\nAnswer with only "entails" or "not entails". No need for extra explanation\nThe answer is:',
   ' not entail'),
  ('Welcome to the "Logic Match Game!"\n\nTask: In this game, your challenge is to decide if the statement, "الريال اليمني يتراجع إلى أدنى مستوى في تاريخه أمام الدولار," logically follows from the premise, "العملة اليمنية عند أدنى مستوى أمام الدولار." Think carefully: does the premise support the hypothesi

In [29]:
# save prompt samples
import json

# create a folder to save the prompts
import os
if not os.path.exists(f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/AceGPT/samples_used_for_tuning'):
    os.makedirs(f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/AceGPT/samples_used_for_tuning')


with open(f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/AceGPT/samples_used_for_tuning/train.json','w') as f:
    json.dump(
        train_samples,
        f,
        indent=4,
        ensure_ascii=False,
    )

with open(f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/AceGPT/samples_used_for_tuning/val.json','w') as f:
    json.dump(
        eval_samples,
        f,
        indent=4,
        ensure_ascii=False,
    )

In [30]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=16,
    eval_batch_size=16,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}'
)

peft config LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=16, target_modules={'q_proj', 'v_proj'}, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
Using auto half precision backend

***** Running Evaluation *****
  Num examples = 500
  Batch size = 16


loading configuration file /hdd/shared_models/AceGPT-7B/config.json
Model config LlamaConfig {
  "_name_or_path": "meta-llama/Llama-2-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_length": 4096,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "float16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 32000
}



{'eval_loss': 3.2303824424743652, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 18.4363, 'eval_samples_per_second': 27.12, 'eval_steps_per_second': 1.736}


***** Running training *****
  Num examples = 4,500
  Num Epochs = 10
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 2,820
  Number of trainable parameters = 8,388,608


...

{'eval_loss': 0.09247545897960663, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 12.3105, 'eval_samples_per_second': 40.616, 'eval_steps_per_second': 2.599, 'epoch': 5.319148936170213}



***** Running Evaluation *****
  Num examples = 500
  Batch size = 16


{'eval_loss': 0.09929891675710678, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 12.3066, 'eval_samples_per_second': 40.629, 'eval_steps_per_second': 2.6, 'epoch': 6.205673758865248}




Training completed. Do not forget to share your model on huggingface.co/models =)




0.03905612975358963

In [ ]:
exit()

: 